# Scopus Column Adaptation

**Pipeline position:** Stage 1 of 3

This notebook merges Scopus export data to add eid, document type, and author keywords for use in the systematic search term identification pipeline.

**Background:** The Uni Mannheim Scopus access expired 2025-12-31. Initial Scopus exports were .txt files, but CSV format was required. The same search results were re-exported from Zotero into CSV files, but they were lacking eid (electronic identifiers), document type, and author keywords. This notebook extracts these fields from the original .txt files and merges them into the CSV exports.

**Input:**
- `01_raw/scopus/exports/*.csv` (Zotero exports)
- `01_raw/scopus/exports/*.txt` (original Scopus exports)

**Output:**
- `02_processed/scopus/*_adapt_columns.csv` (standardized for Stage 2)

In [1]:
import re
import pandas as pd

from pathlib import Path

In [ ]:
# Define input and output directories
input_dir = Path("../01_raw/scopus/exports")
output_dir = Path("../02_processed/scopus")
output_dir.mkdir(parents=True, exist_ok=True)

# Defining file pairs of csv (from zotero and orig. ris export from scopus) and txt
file_pairs = [
    ("scopus_2025-12-29_socialpolicy_crisis.csv", "scopus_2025-12-29_socialpolicy_crisis.txt"),
    ("scopus_2025-12-29_welfarestate_crisis.csv", "scopus_2025-12-29_welfarestate_crisis.txt"),
    ("scopus_2025-12-29_resilience.csv", "scopus_2025-12-29_resilience.txt"),
    ("scopus_2025-12-29_socialpolicy_covid.csv", "scopus_2025-12-29_socialpolicy_covid.txt"),
    ("scopus_2025-12-29_socialpolicy_crisis.csv", "scopus_2025-12-29_socialpolicy_crisis.txt"),
    ("scopus_2025-12-29_socialpolicy_greatrecession.csv", "scopus_2025-12-29_socialpolicy_greatrecession.txt"),
    ("scopus_2025-12-29_welfarestate_covid.csv", "scopus_2025-12-29_welfarestate_covid.txt"),
    ("scopus_2025-12-29_welfarestate_crisis.csv", "scopus_2025-12-29_welfarestate_crisis.txt"),
    ("scopus_2025-12-29_welfarestate_greatrecession.csv", "scopus_2025-12-29_welfarestate_greatrecession.txt"),
    ("scopus_2025-12-29_welfarestatechange.csv", "scopus_2025-12-29_welfarestatechange.txt"),
]


# Loop through each pair
for csv_name, txt_name in file_pairs:
    csv_path = input_dir / csv_name
    txt_path = input_dir / txt_name
    
    print(f"Processing {csv_name} and {txt_name}...")
   
    # 1 read csv
    scopus_df = pd.read_csv(csv_path)

    # 2 load txt
    with open(txt_path, encoding="utf-8") as f:
        text = f.read()

    # 3 split txt into records using "AUTHOR FULL NAMES:" as the anchor
    records = re.split(r'(?=AUTHOR FULL NAMES:)', text)

    rows = []

    for rec in records:
        # extract title (line after the AUTHOR FULL NAMES section)
        title_match = re.search(r'\n([^\n]+)\n\(\d{4}\)', rec)
        if not title_match:
            continue
        title = title_match.group(1).strip()

        # extract eid from url
        eid_match = re.search(r'eid=([0-9\-s\.]+)', rec)
        eid = eid_match.group(1).strip() if eid_match else None

        # extract author keywords
        kw_match = re.search(r'AUTHOR KEYWORDS:\s*(.+)', rec)
        keywords = kw_match.group(1).strip() if kw_match else None

        # extract document type
        dt_match = re.search(r'DOCUMENT TYPE:\s*(.+)', rec)
        doctyp = dt_match.group(1).strip() if kw_match else None

        rows.append({
            "Title": title,
            "eid": eid,
            "keywords": keywords,
            "doctype": doctyp
        })

    txt_df = pd.DataFrame(rows)

    # 4 merge txt_df info into csv based on "Title"
    merged_df = scopus_df.merge(txt_df, on="Title", how="left")

    # 5 renaming columns 
    merged_df = merged_df.rename(columns={
        "Author": "authors",
        "Publication Year": "year",
        "Title": "title",
        "DOI": "doi",
        "Publication Title": "journal",
        "Issue": "issue",
        "Volume": "volume"
    })

    # 6 split the 'Pages' column into "pagestart" and "pageend"
    if 'Pages' in merged_df.columns:
        merged_df[['pagestart', 'pageend']] = merged_df['Pages'].str.split('-', expand=True)

    # 7 restrict to standard columns
    standard_columns = [
        "title", "abstract", "eid", "authors", "journal", "issue",
        "volume", "keywords", "doctype", "year", "doi", "pagestart", "pageend"
    ]
    merged_df = merged_df[[c for c in standard_columns if c in merged_df.columns]]

    # 8 save updated csv
    out_name = Path(csv_name).stem + "_adapt_columns.csv"
    merged_df.to_csv(output_dir / out_name, index=False)

print(f"\nDone! Processed files saved to {output_dir}/")
print("Next step: Run 02_searchterm_identification.ipynb")